In [0]:
%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    total_amount DECIMAL(10, 2)
);

INSERT INTO orders VALUES
(1, 101, '2023-01-15', 250.00),
(2, 102, '2023-01-20', 150.00),
(3, 101, '2023-02-10', 300.00),
(4, 103, '2023-02-15', 200.00),
(5, 102, '2023-03-05', 400.00),
(6, 104, '2023-03-10', 350.00),
(7, 101, '2023-03-15', 500.00),
(8, 103, '2023-04-01', 450.00),
(9, 102, '2023-04-05', 600.00),
(10, 104, '2023-04-10', 700.00);

    

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
WITH ranked_orders AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        total_amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY order_date DESC
        ) AS row_num
    FROM orders
)
SELECT
    order_id,
    customer_id,
    order_date,
    total_amount
FROM ranked_orders
WHERE row_num = 1;

order_id,customer_id,order_date,total_amount
7,101,2023-03-15,500.00
9,102,2023-04-05,600.00
8,103,2023-04-01,450.00
10,104,2023-04-10,700.00


In [0]:
%sql
-- Step 1: Create a Common Table Expression (CTE) named 'ranked_orders'
-- This builds a temporary result set where we assign a ranking to each row.
WITH ranked_orders AS (
    SELECT 
        order_id,
        customer_id,
        order_date,
        total_amount,
        -- ROW_NUMBER() assigns a sequential integer starting at 1.
        -- PARTITION BY groups the data by customer_id so the ranking resets for each customer.
        -- ORDER BY order_date DESC ensures the most recent order gets a row_num of 1.
        ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY order_date DESC
        ) AS row_num
    FROM orders
)
-- Step 2: Query the CTE to filter out everything except the latest record
SELECT 
    order_id,
    customer_id,
    order_date,
    total_amount
FROM ranked_orders
-- Filtering for row_num = 1 isolates the most recent order for every customer.
WHERE row_num = 1;

order_id,customer_id,order_date,total_amount
7,101,2023-03-15,500.00
9,102,2023-04-05,600.00
8,103,2023-04-01,450.00
10,104,2023-04-10,700.00


In [0]:
%sql
SELECT
        order_id,
        customer_id,
        order_date,
        total_amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY order_date DESC
        ) AS row_num
    FROM orders

order_id,customer_id,order_date,total_amount,row_num
7,101,2023-03-15,500.00,1
3,101,2023-02-10,300.00,2
1,101,2023-01-15,250.00,3
9,102,2023-04-05,600.00,1
5,102,2023-03-05,400.00,2
2,102,2023-01-20,150.00,3
8,103,2023-04-01,450.00,1
4,103,2023-02-15,200.00,2
10,104,2023-04-10,700.00,1
6,104,2023-03-10,350.00,2
